In [2]:
import subprocess, os, shutil, sys

# Install packages
subprocess.run([sys.executable, "-m", "pip", "install",
                "py7zr", "duckdb", "pyarrow", "-q"])

# Copy .py files
CODE_DIR = "/kaggle/input/datasets/harshytag/kkbox-project-code/"
WORK_DIR = "/kaggle/working/"
for f in os.listdir(CODE_DIR):
    if f.endswith(".py"):
        shutil.copy(CODE_DIR + f, WORK_DIR + f)

# Patch config.py — fix RAW_DIR path
with open(WORK_DIR + "config.py", "r") as f:
    content = f.read()
content = content.replace(
    'RAW_DIR       = Path("/kaggle/input/kkbox-churn-prediction-challenge")',
    'RAW_DIR       = Path("/kaggle/input/competitions/kkbox-churn-prediction-challenge")'
)
with open(WORK_DIR + "config.py", "w") as f:
    f.write(content)

# Patch verbose=False (removed in newer PyTorch)
for fname in ["utils.py", "05_multitask_ablation.py"]:
    fpath = WORK_DIR + fname
    with open(fpath, "r") as f:
        c = f.read()
    with open(fpath, "w") as f:
        f.write(c.replace(", verbose=False", ""))

# Patch 07_final_evaluation.py — fix expm1 overflow bug
with open(WORK_DIR + "07_final_evaluation.py", "r") as f:
    content = f.read()
content = content.replace(
    'ltv_true_raw = np.expm1(test_df["ltv"].values)',
    'ltv_true_raw = test_df["ltv"].values'
)
with open(WORK_DIR + "07_final_evaluation.py", "w") as f:
    f.write(content)

os.chdir(WORK_DIR)

# Verify
import sys
if "config" in sys.modules:
    del sys.modules["config"]
import config
print("Setup complete")
print(f"Device will be: {'GPU' if __import__('torch').cuda.is_available() else 'CPU'}")
print(f"RAW_DIR exists: {config.RAW_DIR.exists()}")
print(f"Files copied  : {sorted([f for f in os.listdir(WORK_DIR) if f.endswith('.py')])}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.6/495.6 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 13.9 MB/s eta 0:00:00
Setup complete
Device will be: GPU
RAW_DIR exists: True
Files copied  : ['00_data_processing.py', '01_eda.py', '02_feature_engineering.py', '04_training_baselines.py', '05_multitask_ablation.py', '06_calibration_business.py', '07_final_evaluation.py', 'config.py', 'models.py', 'utils.py']


In [3]:
!python 00_data_processing.py

[14:57:42] === Stage 0: Data Processing ===
[14:57:42] Platform : Linux/Kaggle (FIFO enabled)
[14:57:42] RAW_DIR  : /kaggle/input/competitions/kkbox-churn-prediction-challenge
[14:57:42] OUT_DIR  : /kaggle/working/processed
[14:57:42] Extracting members_v3.csv.7z ...
[14:58:04]   -> /kaggle/working/_tmp_extract/members_v3.csv
[14:58:15] Wrote members.parquet (6,769,473 rows)
[14:58:15] Extracting train.csv.7z ...
[14:58:19]   -> /kaggle/working/_tmp_extract/train.csv
[14:58:20] Wrote train.parquet (992,931 rows)
[14:58:20] Extracting train_v2.csv.7z ...
[14:58:23]   -> /kaggle/working/_tmp_extract/data/churn_comp_refresh/train_v2.csv
[14:58:24] Wrote train_v2.parquet (970,960 rows)
[14:58:24] Extracting transactions.csv.7z ...
[14:59:30]   -> /kaggle/working/_tmp_extract/transactions.csv
[14:59:38]   chunk 0 | 5,000,000 rows written so far
[14:59:46]   chunk 1 | 10,000,000 rows written so far
[14:59:55]   chunk 2 | 15,000,000 rows written so far
[15:00:03]   chunk 3 | 20,000,000 rows w

In [4]:
!python 02_feature_engineering.py

=== Stage 2: Feature Engineering & Preprocessing ===
[1/6] Merging tables via DuckDB ...
      Using logs table: user_logs
      Merged: 992,931 rows x 20 cols
[2/6] Engineering derived features ...
[3/6] Train/Val/Test split 70/15/15 (stratified) ...
      train   n=695,051  churn=0.0639
      val     n=148,940  churn=0.0639
      test    n=148,940  churn=0.0639
[4/6] Median imputation (fit on train only) ...
      Medians: {'bd_clean': np.float64(28.0), 'registration_tenure_days': 963.0, 'avg_payment_plan_days': 30.0, 'avg_actual_amount_paid': 139.76923076923077}
[5/6] Label-encoding + StandardScaling (fit on train only) ...
      city                       cardinality: 22
      gender                     cardinality: 4
      registered_via             cardinality: 6
      payment_method_id          cardinality: 36
[6/6] Saving splits + manifest ...
      train : 695,051 rows -> /kaggle/working/processed/model_dataset_train.parquet
      val   : 148,940 rows -> /kaggle/working/proces

In [5]:
!python 04_training_baselines.py

=== Stage 4: Training Baselines (Exp-1 & Exp-2) ===
  Device: cuda
  pos_weight = 14.64  (N_neg=650,621  N_pos=44,430)

--- Exp-1: Churn-Only (lambda_churn=1.0, lambda_ltv=0.0) ---
  Model parameters: 56,130
  [Exp-1] ep  0  tr=1.0139  vl=0.9711  auc=0.8184  rmse=5.3820
  [Exp-1] ep  1  tr=0.9670  vl=0.9450  auc=0.8296  rmse=5.3820
  [Exp-1] ep  2  tr=0.9501  vl=0.9390  auc=0.8325  rmse=5.3820
  [Exp-1] ep  3  tr=0.9424  vl=0.9336  auc=0.8355  rmse=5.3820
  [Exp-1] ep  4  tr=0.9365  vl=0.9288  auc=0.8373  rmse=5.3820
  [Exp-1] ep  5  tr=0.9329  vl=0.9232  auc=0.8395  rmse=5.3820
  [Exp-1] ep  6  tr=0.9297  vl=0.9231  auc=0.8402  rmse=5.3820
  [Exp-1] ep  7  tr=0.9274  vl=0.9206  auc=0.8406  rmse=5.3820
  [Exp-1] ep  8  tr=0.9266  vl=0.9218  auc=0.8402  rmse=5.3820
  [Exp-1] ep  9  tr=0.9250  vl=0.9181  auc=0.8420  rmse=5.3820
  [Exp-1] ep 10  tr=0.9244  vl=0.9228  auc=0.8404  rmse=5.3820
  [Exp-1] ep 11  tr=0.9228  vl=0.9191  auc=0.8414  rmse=5.3820
  [Exp-1] ep 12  tr=0.9230  vl=0.920

In [6]:
!python 05_multitask_ablation.py

=== Stage 5: Multi-Task Ablation (Exp-3 to Exp-7) ===
  Device: cuda
  pos_weight = 14.64  (N_neg=650,621  N_pos=44,430)

--- exp3_fixed_5050 (lc=0.5, ll=0.5) ---
  [exp3_fixed_5050] ep  0  vl=0.7684  auc=0.8071  rmse=0.7412
  [exp3_fixed_5050] ep  1  vl=0.7433  auc=0.8158  rmse=0.7183
  [exp3_fixed_5050] ep  2  vl=0.7389  auc=0.8221  rmse=0.7169
  [exp3_fixed_5050] ep  3  vl=0.7266  auc=0.8277  rmse=0.7073
  [exp3_fixed_5050] ep  4  vl=0.7207  auc=0.8326  rmse=0.7088
  [exp3_fixed_5050] ep  5  vl=0.7092  auc=0.8373  rmse=0.6984
  [exp3_fixed_5050] ep  6  vl=0.7102  auc=0.8367  rmse=0.6992
  [exp3_fixed_5050] ep  7  vl=0.7059  auc=0.8396  rmse=0.6971
  [exp3_fixed_5050] ep  8  vl=0.7004  auc=0.8392  rmse=0.6894
  [exp3_fixed_5050] ep  9  vl=0.7022  auc=0.8405  rmse=0.6924
  [exp3_fixed_5050] ep 10  vl=0.6925  auc=0.8424  rmse=0.6843
  [exp3_fixed_5050] ep 11  vl=0.6972  auc=0.8411  rmse=0.6869
  [exp3_fixed_5050] ep 12  vl=0.6913  auc=0.8421  rmse=0.6815
  [exp3_fixed_5050] ep 13  vl=0

In [7]:
!python 06_calibration_business.py

=== Stage 6: Calibration & Business Decision Layer ===
  Loaded Exp-6 from /kaggle/working/models/exp6_uncertainty.pt

[1] Getting raw logits ...

[2] ECE — uncalibrated ...
  ECE (uncalibrated)     : 0.2736
  Mean predicted P(churn): 0.3375
  True churn rate        : 0.0639
  Cause: pos_weight=14.6 inflates all probabilities.

[3] Temperature scaling ...
  T = 0.8781
  ECE after temp scaling : 0.2586  (barely improved — bias problem)

[4] Isotonic regression ...
  ECE after isotonic     : 0.0008  <- WINNER
  Mean calibrated P      : 0.0637
  Distinct cal probs     : 179 (step function — see project notes on resolution trade-off)

  Reliability diagrams saved.

[5] Building Retention Priority Score ...
  Top 5 users by priority score:
 rank  p_churn      e_ltv  priority_score  is_churn
    1 0.445614 205.827255       91.719513       1.0
    2 0.400376 225.106201       90.127106       1.0
    3 0.445614 202.026184       90.025703       0.0
    4 0.445614 201.412460       89.752220      

In [8]:
!python 07_final_evaluation.py

=== Stage 7: Final Evaluation Summary ===
  Model: Exp-6 (uncertainty weighting) + isotonic calibration

  ┌─────────────────────────────────────────────────────┐
  │            FINAL MODEL METRICS (Test Set)           │
  ├─────────────────────────────────────────────────────┤
  │  churn_auc_roc                      : 0.8454  │
  │  churn_auc_pr                       : 0.3600  │
  │  churn_base_rate                    : 0.0639  │
  │  ltv_rmse_raw_twd                   : 60.6499  │
  │  ltv_mae_raw_twd                    : 32.3144  │
  │  ltv_r2_raw                         : 0.4658  │
  └─────────────────────────────────────────────────────┘

  vs single-task ceilings:
    AUC  : 0.8454 vs 0.8453 (above ceiling)
    RMSE : 60.65 vs 59.51 TWD (worse)

  ROC + PR curves saved.
  LTV scatter saved.

  KKBOX MULTI-TASK MLP — FINAL RESULTS
Model  : Exp-6 Uncertainty Weighting + Isotonic Calibration
Dataset: WSDM 2017 KKBox  |  992,931 users  |  70/15/15 split

CHURN (test):
  AUC-ROC  : 0.

In [9]:
import os

print(f"MODELS: {len(os.listdir('/kaggle/working/models/'))} files")
for f in sorted(os.listdir("/kaggle/working/models/")):
    size = os.path.getsize(f"/kaggle/working/models/{f}") / 1024
    print(f"  {f:40s}  {size:.0f} KB")

print(f"\nRESULTS: {len(os.listdir('/kaggle/working/results/'))} files")
all_ok = True
for f in sorted(os.listdir("/kaggle/working/results/")):
    size = os.path.getsize(f"/kaggle/working/results/{f}") / 1024
    status = "OK" if size > 0 else "EMPTY"
    if size == 0:
        all_ok = False
    print(f"  [{status}]  {f:45s}  {size:.0f} KB")

print(f"\n{'ALL GOOD — Click Save Version now!' if all_ok else 'Some files empty — do not commit yet.'}")

MODELS: 7 files
  exp1_churn_only.pt                        235 KB
  exp2_ltv_only.pt                          235 KB
  exp3_fixed_5050.pt                        235 KB
  exp4_churn_dominant.pt                    235 KB
  exp5_ltv_dominant.pt                      235 KB
  exp6_uncertainty.pt                       235 KB
  exp7_pcgrad.pt                            235 KB

RESULTS: 22 files
  [OK]  ablation_results_table.csv                     0 KB
  [OK]  baseline_results.json                          1 KB
  [OK]  budget_allocation.txt                          0 KB
  [OK]  exp1_history.csv                               4 KB
  [OK]  exp2_history.csv                               3 KB
  [OK]  exp3_fixed_5050_history.csv                    3 KB
  [OK]  exp4_churn_dominant_history.csv                3 KB
  [OK]  exp5_ltv_dominant_history.csv                  3 KB
  [OK]  exp6_history.csv                               5 KB
  [OK]  exp7_history.csv                               3 KB
  [OK]  

In [10]:
import duckdb, os, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set_theme(style="whitegrid")
RESULTS = "/kaggle/working/results/"
con = duckdb.connect()
con.execute("PRAGMA threads=2")
con.execute("PRAGMA memory_limit='4GB'")

TRAIN  = "/kaggle/working/processed/train_v2.parquet"
MEMBER = "/kaggle/working/processed/members.parquet"
TXN    = "/kaggle/working/processed/transactions_v2.parquet"

# Plot 1 — Churn balance
df = con.execute(f"SELECT is_churn, COUNT(*) n FROM '{TRAIN}' GROUP BY 1 ORDER BY 1").df()
df["pct"] = df["n"] / df["n"].sum() * 100
fig, ax = plt.subplots(figsize=(4,3))
ax.bar(df["is_churn"].astype(str), df["n"],
       color=["#42A5F5","#EF5350"])
for i, (n, pct) in enumerate(zip(df["n"], df["pct"])):
    ax.text(i, n+5000, f"{pct:.1f}%", ha="center",
            fontsize=10, fontweight="bold")
ax.set_title("Churn Label Balance (~9% positive class)", fontweight="bold")
ax.set_xlabel("is_churn"); ax.set_ylabel("Count")
fig.tight_layout()
fig.savefig(RESULTS+"eda_churn_balance.png", dpi=150, bbox_inches="tight")
plt.close(); print("Saved: eda_churn_balance.png")

# Plot 2 — Member demographics
members = con.execute(
    f"SELECT bd, gender, registration_init_time FROM '{MEMBER}'"
).df()
valid_bd = members[(members["bd"]>0) & (members["bd"]<=100)]
members["reg_year"] = (members["registration_init_time"]
                       .astype(str).str[:4].astype("Int64"))
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].hist(valid_bd["bd"], bins=50, color="#42A5F5", edgecolor="white")
axes[0].set_title(f"Age (valid 1-100: {len(valid_bd)/len(members)*100:.0f}%)")
gc = members["gender"].fillna("unknown").value_counts()
axes[1].bar(gc.index, gc.values, color=["#42A5F5","#EF5350","#BDBDBD"])
axes[1].set_title(f"Gender ({members['gender'].isna().mean()*100:.0f}% unknown)")
yr = (members[members["reg_year"].between(2004,2017)]["reg_year"]
      .value_counts().sort_index())
axes[2].bar(yr.index.astype(str), yr.values, color="#66BB6A")
axes[2].set_title("Registrations by year")
axes[2].tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(RESULTS+"eda_member_demographics.png", dpi=150, bbox_inches="tight")
plt.close(); print("Saved: eda_member_demographics.png")

# Plot 3 — Churn by attributes
by_gender = con.execute(f"""
    SELECT COALESCE(m.gender,'unknown') gender,
           COUNT(*) n, AVG(t.is_churn)*100 churn_pct
    FROM '{TRAIN}' t LEFT JOIN '{MEMBER}' m USING (msno) GROUP BY 1
""").df()
by_channel = con.execute(f"""
    SELECT m.registered_via, COUNT(*) n, AVG(t.is_churn)*100 churn_pct
    FROM '{TRAIN}' t LEFT JOIN '{MEMBER}' m USING (msno)
    GROUP BY 1 ORDER BY n DESC LIMIT 6
""").df()
by_age = con.execute(f"""
    SELECT CASE WHEN m.bd BETWEEN 1  AND 17  THEN '1-17'
                WHEN m.bd BETWEEN 18 AND 24  THEN '18-24'
                WHEN m.bd BETWEEN 25 AND 34  THEN '25-34'
                WHEN m.bd BETWEEN 35 AND 44  THEN '35-44'
                WHEN m.bd BETWEEN 45 AND 100 THEN '45-100'
                ELSE 'unknown' END age_bucket,
           COUNT(*) n, AVG(t.is_churn)*100 churn_pct
    FROM '{TRAIN}' t LEFT JOIN '{MEMBER}' m USING (msno)
    GROUP BY 1 ORDER BY n DESC
""").df()
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].bar(by_gender["gender"], by_gender["churn_pct"], color="#42A5F5")
axes[1].bar(by_channel["registered_via"].astype(str),
            by_channel["churn_pct"], color="#42A5F5")
axes[2].bar(by_age["age_bucket"], by_age["churn_pct"], color="#42A5F5")
axes[2].tick_params(axis="x", rotation=30)
for ax, ttl in zip(axes, ["by gender","by channel","by age"]):
    ax.set_title(f"Churn rate {ttl}"); ax.set_ylabel("Churn rate (%)")
fig.tight_layout()
fig.savefig(RESULTS+"eda_churn_by_attributes.png", dpi=150, bbox_inches="tight")
plt.close(); print("Saved: eda_churn_by_attributes.png")

# Plot 4 — Transaction signals
txn = con.execute(f"""
    WITH latest AS (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY msno ORDER BY transaction_date DESC) rn
        FROM '{TXN}'
    )
    SELECT l.is_auto_renew, l.is_cancel,
           COUNT(*) n, ROUND(AVG(t.is_churn)*100,1) churn_pct
    FROM latest l JOIN '{TRAIN}' t USING (msno)
    WHERE l.rn=1 GROUP BY 1,2 ORDER BY 1,2
""").df()
txn["segment"] = txn.apply(
    lambda r: f"renew={int(r.is_auto_renew)}\ncancel={int(r.is_cancel)}", axis=1)
fig, ax = plt.subplots(figsize=(7,3.5))
bars = ax.bar(txn["segment"], txn["churn_pct"],
              color=["#66BB6A","#42A5F5","#FFA726","#EF5350"])
for bar, pct in zip(bars, txn["churn_pct"]):
    ax.text(bar.get_x()+bar.get_width()/2,
            bar.get_height()+0.5, f"{pct:.1f}%",
            ha="center", fontsize=9, fontweight="bold")
ax.set_title("Churn rate by auto-renew / cancel status")
ax.set_ylabel("Churn rate (%)")
fig.tight_layout()
fig.savefig(RESULTS+"eda_transaction_signals.png", dpi=150, bbox_inches="tight")
plt.close(); print("Saved: eda_transaction_signals.png")

print("\nAll 4 EDA plots saved!")

Saved: eda_churn_balance.png
Saved: eda_member_demographics.png
Saved: eda_churn_by_attributes.png
Saved: eda_transaction_signals.png

All 4 EDA plots saved!
